In [5]:
import numpy as np
import yfinance as yf
from statsmodels.tools import add_constant
from statsmodels.regression.linear_model import OLS

In [43]:
# Testing on Cola And Pepsi
data = yf.download(["KO","PEP"], start="2015-10-01", end="2020-01-01")["Close"]
data.columns = data.columns.get_level_values(0)
data = data.dropna()

[*********************100%***********************]  2 of 2 completed


In [44]:
ko = data["KO"]
pep = data["PEP"]

In [45]:
# Hedge Ratio
model = OLS(ko, add_constant(pep)).fit()
hedge_ratio = model.params["PEP"]

In [46]:
# Spread and zscore
spread = ko - hedge_ratio * pep
z_score = (spread - spread.mean())/spread.std()

In [47]:
# signals
longEntry  = z_score < -1
longExit   = z_score >= 0
shortEntry = z_score > 1
shortExit  = z_score <= 0

In [48]:
# Positions
position = np.zeros(len(z_score))
for t in range(1,len(z_score)):
    if position[t-1] == 0:
        if longEntry.iloc[t]:
            position[t] = 1
        elif shortEntry.iloc[t]:
            position[t] = -1
    elif position[t-1] == 1:
        if longExit.iloc[t]:
            position[t] = 0
        else:
            position[t] = 1
    elif position[t-1] == -1:
        if shortExit.iloc[t]:
            position[t] = 0
        else:
            position[t] = -1

In [49]:
# Daily returns 
koRet = ko.pct_change().fillna(0)
pepRet = pep.pct_change().fillna(0)

dailyRet = np.where(
    position == 1, (koRet - pepRet) / 2,
    np.where(
        position == -1, (pepRet - koRet) / 2,0
    )
)

In [53]:
#  Kelly Inputs
mu = np.mean(dailyRet) # mean daily return
variance = np.var(dailyRet)
r = 0.04/252 # daily risk free rate (4% annual)

In [54]:
# Kellys fraction
f = (mu - r) / variance

In [55]:
print(f"Mean Daily Return (μ):   {mu:.6f}")
print(f"Daily Variance (σ²):     {variance:.6f}")
print(f"Daily Risk Free Rate (r):{r:.6f}")
print(f"Kelly Fraction (f*):     {f:.4f}")
print(f"As percentage:           {f*100:.2f}%")

Mean Daily Return (μ):   0.000083
Daily Variance (σ²):     0.000009
Daily Risk Free Rate (r):0.000159
Kelly Fraction (f*):     -8.9258
As percentage:           -892.58%
